# Doc2Vec (Paragraph Vector)

## Goal

Huấn luyện Doc2Vec (Le & Mikolov, 2014) trên cùng corpus mẫu đã dùng ở `notebook/phoBERT` để so sánh cách tiếp cận embedding cổ điển (Doc2Vec, không cần transformer) với embedding dựa trên PhoBERT.

Doc2Vec mở rộng Word2Vec bằng cách học thêm một vector cho mỗi "paragraph" (ở đây là mỗi câu) song song với vector từ:

- **PV-DM** (`dm=1`): giống CBOW — dùng paragraph vector + context words để dự đoán từ tiếp theo.
- **PV-DBOW** (`dm=0`): giống Skip-gram — chỉ dùng paragraph vector để dự đoán các từ xuất hiện trong đoạn.

Notebook này tự cài `gensim` bằng `%pip` ở cell đầu tiên nên không cần sửa `pyproject.toml` của backend. Toàn bộ output (model, embedding) được lưu trong `notebook/Doc2Vec/artifacts/`, tách biệt khỏi `backend/data/` dùng chung cho pipeline PhoBERT.

## Setup

Chạy notebook từ thư mục `backend/` (giống quy ước của `notebook/phoBERT`) để `torch` đã có sẵn từ dependency của backend; `gensim` được cài riêng ở cell dưới vì backend chưa khai báo dependency này.

**Đã kiểm tra thực tế:** `gensim==4.4.0` (bản mới nhất tại thời điểm viết) **không cài được trên Python 3.14** — backend `pyproject.toml` pin `requires-python = ">=3.14,<3.15"`, nhưng PyPI chưa có wheel `cp314` cho `gensim`, và bản sdist chỉ đóng gói sẵn file `.c` đã Cython-hoá bằng bản Cython cũ, dùng field `ma_version_tag` trên `PyDictObject` — field này đã bị CPython 3.14 xoá khỏi C-API nội bộ nên build từ source cũng lỗi ngay ở bước compile (`fatal error: too many errors emitted`). Sdist cũng không kèm file `.pyx` gốc nên không thể ép Cython build lại bằng tay.

Notebook này (và `02`, `03` trong cùng thư mục) đã được chạy thử thành công trên **Python 3.11** (`gensim==4.4.0` cài bằng wheel bình thường, `vocab size = 307`, model lưu ra `artifacts/doc2vec_dm.model` và `doc2vec_dbow.model` đúng như mong đợi). Vì backend dùng chung một `.venv` 3.14 cho FastAPI + PhoBERT, khuyến nghị: tạo riêng một virtualenv Python 3.11–3.13 chỉ để chạy 3 notebook Doc2Vec này (chọn kernel tương ứng trong Jupyter/VSCode), thay vì dùng chung kernel 3.14 của `backend/.venv`. Khi PyPI có wheel `cp314` cho `gensim` (hoặc gensim phát hành bản tương thích CPython 3.14), có thể quay lại dùng chung kernel 3.14 và bỏ ghi chú này.

In [ ]:
%pip install -q "gensim>=4.3,<5.0"

In [ ]:
from pathlib import Path

from gensim.models.doc2vec import Doc2Vec, TaggedDocument

## Corpus

Dùng lại đúng bài báo mẫu (34 câu, đã word-segmented bằng VnCoreNLP) từ `phoBERT/03_document_embedding_v2.ipynb` để hai phương pháp có thể so sánh trên cùng dữ liệu. Doc2Vec cần nhiều "document" mới học được vector có ý nghĩa, nên ở đây mỗi câu được coi là một document riêng, đánh tag theo chỉ số.

In [ ]:
# ruff: noqa: E501
segmented_sentences = [
    'Đặt vé từ TP HCM đi Singapore để công_tác , chị Hoàng_Loan , ở phường Xuân_Hoà , bất_ngờ vì mức giá lần đầu mua được kể từ sau đại_dịch " Năm_ngoái , chặng TP HCM - Singapore có lúc lên tới 3,2 triệu đồng một_chiều , còn năm nay tôi chỉ trả hơn 1,6 triệu đồng , đã gồm thuế , phí " , chị nói .',
    "Chiều về , vé cũng được áp_dụng mức giá khuyến_mại 19.000 đồng , nhưng sau khi cộng thuế , phí , tổng tiền chị Loan phải trả khoảng 2,4 triệu đồng .",
    "Theo chị , các khoản phí tại sân_bay Singapore cao hơn chiều bay từ Việt_Nam nên dù cùng giá vé niêm_yết , số tiền thực trả vẫn chênh_lệch đáng_kể .",
    "Tính cả hai chiều , chuyến đi Singapore của chị hết hơn 4 triệu đồng , giảm khoảng một_nửa so với cùng kỳ năm_ngoái .",
    "Sau Covid-19 , các đường_bay quốc_tế mất nhiều thời_gian để phục_hồi , trong khi nguồn cung chưa trở_lại như trước khiến giá luôn ở mức cao , nhất_là vào mùa du_lịch .",
    "Năm nay , nguồn cung tăng nhanh hơn , kéo_theo cạnh_tranh giữa các hãng và tạo thêm dư_địa giảm_giá .",
    "Mức giá chị Loan mua không phải trường_hợp cá_biệt .",
    "Khảo_sát các đường_bay từ TP HCM đi Singapore và Thái_Lan cho thấy mức giá khuyến_mại 19.000-90.000 đồng , chưa gồm thuế , phí chiếm đa_số các chặng bay trong tháng 8 và 9 .",
    "Sau khi cộng các khoản này , vé TP HCM - Singapore từ hơn 1,6 triệu đồng một_chiều , còn chặng TP HCM - Bangkok chưa đến 1,9 triệu đồng .",
    "Vé của một_số hãng hàng_không nước_ngoài trên cùng_đường bay hiện cao hơn khoảng 2-3 lần so với các hãng Việt_Nam .",
    "Từ Hà_Nội đi Singapore và Thái_Lan , giá vé của các hãng trong nước dao_động 2,6-3 triệu đồng một_chiều , đã gồm thuế , phí .",
    "Một_số ngày trong tháng 8 , mức thấp nhất còn hơn 2 triệu đồng .",
    "Với đường_bay TP HCM - Jakarta , giá cũng giảm nhưng mặt_bằng vẫn cao hơn Singapore và Thái_Lan .",
    "Nếu trước_đây vé khứ_hồi thường ở mức 7-10 triệu đồng , hiện giá thấp nhất khoảng 6,3 triệu đồng , đã gồm thuế , phí , tương_đương hơn 3 triệu đồng mỗi chiều .",
    "Mức giá cao hơn một phần do quãng đường_bay xa hơn .",
    "Các đường_bay từ Hà_Nội và TP HCM tới châu_Âu , Đông_Bắc_Á cũng giảm khoảng 10-15% so với trước .",
    "Giá đi xuống trong bối_cảnh nguồn cung hàng_không Việt_Nam tăng .",
    "Theo dữ_liệu dự_báo của Công_ty cung_cấp dữ_liệu hàng không OAG ( Anh ) , Việt_Nam có khoảng 7,3 triệu ghế cung_ứng trong tháng 8 , tăng 10% so với cùng kỳ năm_ngoái và đứng thứ hai Đông_Nam_Á , sau Indonesia .",
    "Trong khi tổng năng_lực khai_thác của thị_trường hàng_không Đông_Nam_Á tháng 8 chỉ tăng 0,8% so với cùng kỳ , nguồn cung của Việt_Nam tăng tới 10% .",
    "Trong đó , Vietnam_Airlines có khoảng 2,8 triệu ghế , tăng 8,2% , trong khi Vietjet khoảng 2,24 triệu ghế .",
    "Nguồn cung trên các đường_bay quốc_tế cũng được tăng_cường .",
    "Vietjet_Air cho biết , nâng tần_suất TP HCM - Kuala_Lumpur lên 7 chuyến mỗi tuần trong mùa cao_điểm , đồng_thời mở đường_bay TP HCM - Colombo từ ngày 18/8 .",
    "Hãng cũng chuẩn_bị khai_thác các đường_bay Hà_Nội - Almaty và Hà_Nội - Praha từ tháng 10 .",
    "Ông Hồng_Thanh , chủ một đại_lý vé máy_bay tại TP HCM , cho biết nguồn cung tăng và cạnh_tranh giữa các hãng là nguyên_nhân quan_trọng khiến giá vé quốc_tế hạ nhiệt .",
    "Các hãng phải tăng khuyến_mại , kích_cầu trong bối_cảnh sức_mua chưa phục_hồi như kỳ_vọng .",
    "Chi_phí nhiên_liệu cũng thuận_lợi hơn cho các hãng .",
    "Từ ngày 1/7 , Chính_phủ tiếp_tục kéo_dài thời_hạn áp_dụng thuế nhập_khẩu ưu_đãi , thuế bảo_vệ môi_trường và thuế_giá_trị gia_tăng với xăng_dầu , nhiên_liệu bay đến hết ngày 30/9/2026 , giúp giảm một phần chi_phí đầu_vào của các hãng hàng_không .",
    "Về nhu_cầu , thị_trường khách quốc_tế đến Việt_Nam tăng mạnh .",
    "Bảy tháng đầu năm , Việt_Nam đón gần 14 triệu lượt khách quốc_tế , tăng gần 14% so với cùng kỳ năm_ngoái .",
    "Riêng tháng 7 , lượng khách đạt khoảng 1,67 triệu lượt , trong đó đường_hàng không chiếm gần 83% .",
    "Nhu_cầu đi_lại quốc_tế tăng trong khi nguồn cung được bổ_sung khiến các hãng phải cạnh_tranh mạnh hơn để thu_hút khách .",
    "Đây cũng là một trong những yếu_tố kéo mặt_bằng giá xuống trong mùa hè năm nay .",
    "Không_chỉ quốc_tế , trước đó các hãng cũng liên_tục kích_cầu trên thị_trường nội_địa ngay giữa cao_điểm hè .",
    "Nhiều chương_trình đưa giá vé một_số chặng về mức 0 đồng hoặc vài chục nghìn đồng , chưa gồm thuế , phí .",
]

print(f"Number of sentences: {len(segmented_sentences)}")

### Tokenize

Corpus đã qua VnCoreNLP nên từ ghép được nối bằng `_` (ví dụ `công_tác`); tokenize ở đây chỉ cần tách theo khoảng trắng để giữ nguyên các từ ghép làm một token duy nhất.

In [ ]:
def tokenize(sentence: str) -> list[str]:
    """Tách theo khoảng trắng; corpus đã segment nên giữ nguyên từ ghép nối bằng '_' làm một token."""
    return sentence.split()


tagged_documents = [
    TaggedDocument(words=tokenize(sentence), tags=[str(index)])
    for index, sentence in enumerate(segmented_sentences)
]

print(tagged_documents[0])

## Train PV-DM và PV-DBOW

Huấn luyện cả hai kiến trúc để so sánh ở bước similarity. `vector_size=768` được chọn bằng với PhoBERT chỉ để hai không gian vector có cùng số chiều khi so sánh ở notebook `03_similarity.ipynb` — không có nghĩa là cùng scale hay cùng ngữ nghĩa, vì Doc2Vec ở đây được train từ đầu trên một corpus rất nhỏ (34 pseudo-document).

In [ ]:
VECTOR_SIZE = 768
WINDOW = 5
MIN_COUNT = 1
EPOCHS = 200
SEED = 42


def train_doc2vec(dm: int) -> Doc2Vec:
    """dm=1 train theo PV-DM (CBOW-style); dm=0 train theo PV-DBOW (Skip-gram-style)."""
    model = Doc2Vec(
        vector_size=VECTOR_SIZE,
        window=WINDOW,
        min_count=MIN_COUNT,
        dm=dm,
        workers=1,
        seed=SEED,
        epochs=EPOCHS,
    )
    model.build_vocab(tagged_documents)
    model.train(tagged_documents, total_examples=model.corpus_count, epochs=model.epochs)
    return model


dm_model = train_doc2vec(dm=1)
dbow_model = train_doc2vec(dm=0)

print(f"PV-DM vocab size: {len(dm_model.wv)}")
print(f"PV-DBOW vocab size: {len(dbow_model.wv)}")

## Save models

Lưu trong `notebook/Doc2Vec/artifacts/` (thư mục mới, không đụng tới `backend/data/` đang dùng cho PhoBERT).

In [ ]:
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

DM_PATH = ARTIFACT_DIR / "doc2vec_dm.model"
DBOW_PATH = ARTIFACT_DIR / "doc2vec_dbow.model"

dm_model.save(str(DM_PATH))
dbow_model.save(str(DBOW_PATH))

print(f"Saved: {DM_PATH}")
print(f"Saved: {DBOW_PATH}")

## Next Steps

- `02_document_embedding.ipynb`: dùng model đã train để lấy sentence vector (`model.dv`) và document vector (`model.infer_vector`).
- `03_similarity.ipynb`: so sánh ranking của Doc2Vec với cách làm ở `phoBERT/04_similarity.ipynb` trên cùng một query.
- Corpus thật (`make crawl` + `make preprocess` + `make segment`) sẽ cho nhiều document hơn hẳn 34 câu ở đây; khi đó nên train Doc2Vec theo từng bài báo là một document thay vì từng câu.